# Recipe Analysis and Costing Agent


## Phase 0: Instructions & Setup

---
### **Workflow Rules: Please Read Before Running**

**This notebook is interactive. Follow these rules to avoid errors and unnecessary API calls.**

#### A) First Time Running or After Restarting:
1.  **Run ALL cells in order from top to bottom.** This is required to load data and build the variables needed for later steps.

#### B) After Changing a `.py` File in the `/src` Directory:
1.  **Do NOT restart the kernel.**
2.  **Run ONLY the "Phase 0" Setup Cell below.** (The one with `%autoreload 2`). This loads your changes.
3.  **SKIP to the specific Phase you want to test** (e.g., Phase 2 or 4) and run that cell directly. This avoids re-running earlier phases and saves API costs.

---


In [ ]:
# This cell enables autoreload. Run it once when you start, and re-run it
# anytime you change a .py file in the /src directory.
%load_ext autoreload
%autoreload 2

# This allows us to import from the /src directory
import sys
import os
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print("✅ Setup complete. Autoreload is active.")


## Phase 1: Load Initial Data


In [ ]:
import pandas as pd
import os

# --- Configuration ---
DATA_DIR = os.path.join('..', 'data')
INPUT_FILE = os.path.join(DATA_DIR, 'input', 'RECETARIO.xlsx')

# --- Action ---
print("Recipe Analysis Agent: Initializing...")
print(f"Attempting to load data from: {INPUT_FILE}")

try:
    df = pd.read_excel(INPUT_FILE)
    print("✅ Data loaded successfully.")
    print(f"Found {df.shape[0]} rows and {df.shape[1]} columns.")
    print("\nHere is a preview of your data:")
    display(df.head())
except FileNotFoundError:
    print(f"❌ ERROR: File not found at '{INPUT_FILE}'.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred: {e}")


## Phase 2: AI-Powered Hierarchy Labeling


In [ ]:
import json
from src.prompt_engineering import create_hierarchy_labeling_prompt
from src.gemini_connector import call_gemini_api

# --- 1. Gather Context for the AI ---
print("--- 1. Gathering Context for AI ---")
unique_products = df['PRODUCTO'].dropna().unique().tolist()
print(f"Found {len(unique_products)} unique products to classify.")
df_sample_string = df[['PRODUCTO', 'INGREDIENTES']].head().to_string(index=False)

# --- 2. AI Classification Proposal ---
print("\n--- 2. AI Classification Proposal ---")
prompt = create_hierarchy_labeling_prompt(unique_products, df_sample_string)
ai_response_str = call_gemini_api(prompt)

# --- 3. Parse and Apply Labels ---
print("\n--- 3. Parsing and Applying Labels ---")
try:
    cleaned_response = ai_response_str.strip().replace('```json', '').replace('```', '')
    product_classifications = json.loads(cleaned_response)
    
    classification_df = pd.DataFrame(list(product_classifications.items()), columns=['Product', 'Proposed Type'])
    print("🤖 AI Proposal Received:")
    display(classification_df)
    
    df['HIERARCHY_TYPE'] = df['PRODUCTO'].map(product_classifications)
    print("\n✅ New 'HIERARCHY_TYPE' column added. Here's a preview:")
    display(df.head())
except Exception as e:
    print(f"❌ ERROR: Could not parse or apply AI response.")
    print(f"Raw Response: {ai_response_str}")
    print(f"Error details: {e}")


## Phase 3: Export for Hierarchy Review


In [ ]:
# --- Export for Human Review ---
REVIEW_FILE_PATH = os.path.join(DATA_DIR, 'output', 'RECETARIO_with_hierarchy_for_review.xlsx')
try:
    df.to_excel(REVIEW_FILE_PATH, index=False)
    print("✅ Successfully exported the data for your review.")
    print(f"File saved at: {REVIEW_FILE_PATH}")
    print("\nACTION REQUIRED: Open the file, review the 'HIERARCHY_TYPE' column, make corrections, and save.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred: {e}")


## Phase 4: Load Validated Data & Propose Ingredient Standardization


In [ ]:
import time
from src.prompt_engineering import create_name_standardization_prompt
from src.gemini_connector import call_gemini_api
from src.utils import extract_json_from_string

# --- 1. Load the reviewed file ---
REVIEWED_FILE_PATH = os.path.join(DATA_DIR, 'output', 'RECETARIO_with_hierarchy_for_review.xlsx')
print(f"--- 1. Loading your reviewed file from: {REVIEWED_FILE_PATH} ---")
try:
    validated_df = pd.read_excel(REVIEWED_FILE_PATH)
    # --- Clean column names to prevent errors ---
    validated_df.columns = validated_df.columns.str.strip().str.upper()
    print("✅ Successfully loaded and cleaned column names.")
except Exception as e:
    print(f"❌ ERROR: Could not load the reviewed file: {e}")
    raise

# --- 2. Batch Processing for AI Standardization ---
print("\n--- 2. Getting AI Standardization Proposal in Batches ---")
unique_ingredients = validated_df['INGREDIENTES'].dropna().unique().tolist()
ingredient_name_mappings = {}
CHUNK_SIZE = 50  # Process 50 ingredients at a time
num_chunks = (len(unique_ingredients) + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(0, len(unique_ingredients), CHUNK_SIZE):
    chunk = unique_ingredients[i:i + CHUNK_SIZE]
    chunk_num = (i // CHUNK_SIZE) + 1
    print(f"\nProcessing chunk {chunk_num} of {num_chunks}...")
    
    prompt = create_name_standardization_prompt(chunk)
    ai_response_str = call_gemini_api(prompt)
    
    # Use our robust function to extract the JSON from the response for this chunk
    chunk_mapping = extract_json_from_string(ai_response_str)
    
    if chunk_mapping:
        ingredient_name_mappings.update(chunk_mapping)
        print(f"✅ Successfully processed chunk {chunk_num}.")
    else:
        print(f"⚠️ WARNING: Failed to process chunk {chunk_num}. The AI response was not valid JSON.")
        print(f"Raw Response for this chunk:\n---\n{ai_response_str}\n---")
    
    # Be respectful of API rate limits
    time.sleep(1)

print(f"\n✅ Finished processing all chunks. {len(ingredient_name_mappings)} mappings were created.")

# --- 3. Apply the combined mappings to the DataFrame ---
print("\n--- 3. Applying Full AI Proposal to DataFrame ---")
try:
    if not ingredient_name_mappings:
        raise ValueError("The AI processing resulted in no valid name mappings.")

    validated_df['INGREDIENTE_STANDARDIZED'] = validated_df['INGREDIENTES'].map(ingredient_name_mappings)
    
    print("✅ New 'INGREDIENTE_STANDARDIZED' column added.")
    display(validated_df.head())
    
except Exception as e:
    print(f"❌ ERROR: Could not apply AI response. Details: {e}")
    raise

# --- 4. Export the Result for Review ---
print("\n--- 4. Exporting for Final Review ---")
FINAL_REVIEW_PATH = os.path.join(DATA_DIR, 'output', 'RECETARIO_final_review.xlsx')
try:
    review_columns = ['HIERARCHY_TYPE', 'PRODUCTO', 'INGREDIENTES', 'INGREDIENTE_STANDARDIZED', 'MEDIDA', 'CANTIDAD']
    validated_df[review_columns].to_excel(FINAL_REVIEW_PATH, index=False)
    
    print(f"✅ Successfully exported data for your final review.")
    print(f"File saved at: {FINAL_REVIEW_PATH}")
    print("\nACTION REQUIRED: Open the file, review, make corrections, and save.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred while exporting: {e}")


In [ ]:
## Phase 5: Load Validated Data & Propose SKU Matching


In [ ]:
import pandas as pd
import time
from src.database_connector import get_approved_skus_df
from src.prompt_engineering import create_sku_matching_prompt
from src.gemini_connector import call_gemini_api
from src.utils import extract_json_from_string

# --- 1. Load the final reviewed data ---
print("--- 1. Loading Final Human-Validated Data ---")
FINAL_REVIEW_PATH = os.path.join(DATA_DIR, 'output', 'RECETARIO_final_review.xlsx')
try:
    final_df = pd.read_excel(FINAL_REVIEW_PATH)
    final_df.columns = final_df.columns.str.strip().str.upper()
    print("✅ Successfully loaded the final reviewed data.")
except Exception as e:
    print(f"❌ ERROR: Could not load the final reviewed file: {e}")
    raise

# --- 2. Fetch the Approved SKUs from the database ---
approved_skus_df = get_approved_skus_df()
if approved_skus_df.empty:
    raise ValueError("Failed to load approved SKUs from the database. Cannot proceed.")
print(f"Database columns found: {approved_skus_df.columns.tolist()}")

# --- 3. AI-Powered SKU Matching in Batches ---
print("\n--- 3. Matching Ingredients to SKUs using AI ---")
if 'INGREDIENTE_STANDARDIZED' not in final_df.columns:
    raise KeyError("The required column 'INGREDIENTE_STANDARDIZED' was not found in the final review file.")
    
unique_ingredients = final_df['INGREDIENTE_STANDARDIZED'].dropna().unique()
sku_matches = {}

# Use the CORRECT, cleaned, lowercase column names from the database for the AI's context
# Using sku_key, normalized_description, and sub_sub_category
sku_list_string = approved_skus_df[['sku_key', 'normalized_description', 'sub_sub_category']].to_string(index=False)

for ingredient in unique_ingredients:
    print(f"Processing ingredient: '{ingredient}'...")
    
    prompt = create_sku_matching_prompt(ingredient, sku_list_string)
    ai_response_str = call_gemini_api(prompt)
    
    match_json = extract_json_from_string(ai_response_str)
    
    if match_json and 'best_match_sku' in match_json:
        sku_matches[ingredient] = str(match_json['best_match_sku']).lower()
    else:
        sku_matches[ingredient] = "no_match_found"
    
    time.sleep(1)

print("\n✅ Finished matching all ingredients.")

# --- 4. Merge SKU Data into the Final DataFrame ---
print("\n--- 4. Merging SKU data into recipes ---")
try:
    final_df['matched_sku'] = final_df['INGREDIENTE_STANDARDIZED'].map(sku_matches)
    approved_skus_df['matched_sku'] = approved_skus_df['sku_key'].str.lower()

    # Use the CORRECT, cleaned, lowercase column names for the merge
    # Note: There is no 'unit_price' column, using 'typical_quantity_range' as a placeholder for price.
    sku_details_to_merge = approved_skus_df[[
        'matched_sku', 'category', 'subcategory', 'sub_sub_category',
        'standardized_unit', 'typical_quantity_range', 'last_used'
    ]].rename(columns={
        'category': 'sku_category',
        'subcategory': 'sku_sub_category',
        'sub_sub_category': 'sku_sub_sub_category',
        'standardized_unit': 'sku_unit',
        'typical_quantity_range': 'sku_price', # Placeholder for price
        'last_used': 'sku_last_purchase'
    })
    
    enriched_df = pd.merge(final_df, sku_details_to_merge, on='matched_sku', how='left')
    
    print("✅ Successfully merged SKU details.")
    display(enriched_df.head())

except Exception as e:
    print(f"❌ ERROR: An error occurred while merging SKU data: {e}")
    raise

# --- 5. Export the Final, Enriched Data ---
print("\n--- 5. Exporting final data with costs ---")
COSTING_FILE_PATH = os.path.join(DATA_DIR, 'output', 'RECETARIO_with_costs.xlsx')
try:
    enriched_df.to_excel(COSTING_FILE_PATH, index=False)
    print(f"✅ FINAL SUCCESS! The fully enriched recipe file is saved at:")
    print(COSTING_FILE_PATH)
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred while exporting the final file: {e}")


In [ ]:
## Phase 6: Load Validated SKU MATCHING AND ASSIGN SUB RECEPIES TO NON IDENTIFIED INGREDIENTS
